In [ ]:
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "jupytergis==0.16.8",
#     "openeo",
# ]
# ///

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and then launch this notebook with:

```
uvx juv run ndsi_sentinel2.ipynb
```

(The `uvx` command is included when installing `uv`).

## License

This notebook is an adaptation of the openEO-UDP notebook
[ndsi.ipynb](https://github.com/developmentseed/openeo-udp/blob/main/notebooks/sentinel/sentinel-2/land-cover/ndsi.ipynb)
by Development Seed. That notebook is a conversion of a Sentinel Hub evalscript and is licensed under **CC-BY-SA-4.0**.
This adaptation is also licensed under **CC-BY-SA-4.0**.

- Original evalscript: https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/ndsi/
- Source: Sentinel Hub Custom Scripts (CC-BY-SA-4.0)
- Conversion: Development Seed (openEO-UDP project)

# Normalized Difference Snow Index (NDSI) over Haka Conservation Park, New Zealand, September 2019

This notebook detects snow in the Southern Alps of New Zealand (around Haka Conservation Park) with the Normalized Difference Snow Index (NDSI) from Sentinel-2 L2A data with openEO.
It shows the result as a dynamic tile layer in a JupyterGIS map, on top of an OpenStreetMap basemap.

$$
NDSI = \frac{B03 - B11}{B03 + B11}
$$

Snow is highly reflective in the visible bands (green, B03) but strongly absorbs short-wave infrared (SWIR, B11), so it has a high NDSI.
Clouds stay bright in both the visible and the SWIR, so NDSI separates snow from most clouds.

The steps are:
1. Connect to the openEO backend.
2. Load the Sentinel-2 L2A scenes from 19 to 21 September 2019 and make a first-valid-pixel mosaic.
3. Calculate the NDSI and classify pixels with NDSI > 0.42 as snow.
4. Show snow pixels in blue and the other pixels as a brightened true color composite.
5. Show the result in JupyterGIS.

## Connect to the openEO backend

In [ ]:
import openeo
from openeo.processes import array_create, if_

connection = openeo.connect("https://openeo.mpc.ds.io/")

In [ ]:
connection.authenticate_oidc()

## Parameters

The bounding box covers the mountains between Lake Wakatipu and Lake Wanaka, in the Southern Alps of New Zealand, at the end of the austral winter.

The time range is 19 to 21 September 2019. The end date is exclusive in openEO.
The cloud cover filter keeps only scenes with 30 % cloud cover or less.

`ndsi_threshold` is the NDSI value above which a pixel is classified as snow.

In [ ]:
location_name = "Haka Conservation Park, New Zealand"

bounding_box = {"west": 168.273, "south": -45.589, "east": 169.097, "north": -45.121}
temporal_extent = ["2019-09-19", "2019-09-21"]
max_cloud_cover = 30
ndsi_threshold = 0.42

# Sentinel-2 bands for the algorithm, in this order:
# B02 (blue), B03 (green), B04 (red), B11 (SWIR 1610 nm)
bands = ["B02", "B03", "B04", "B11"]

## Load the Sentinel-2 data

The algorithm needs the visible bands for the true color composite, and the green and SWIR bands for the NDSI.

- **B02**: Blue, 492.4 nm (true color)
- **B03**: Green, 559.8 nm (NDSI, true color)
- **B04**: Red, 664.6 nm (true color)
- **B11**: SWIR, 1613.7 nm (NDSI)

In [ ]:
s2cube = connection.load_collection(
    "sentinel-2-l2a",
    spatial_extent=bounding_box,
    temporal_extent=temporal_extent,
    bands=bands,
    properties={"eo:cloud_cover": lambda x: x <= max_cloud_cover},
)

s2cube = s2cube.process(
    "apply_pixel_selection",
    pixel_selection="first",
    data=s2cube,
)

## NDSI snow classification

Pixels with `NDSI > ndsi_threshold` (0.42) are classified as snow.

| Class | Color |
|---|---|
| Snow | Blue (R=0, G=0.8, B=1.0) |
| Not snow | True color (B04, B03, B02), brightened with a gain of 2.5 |

The backend returns reflectance values from 0 to 1, so the bands are used without scaling.

In [ ]:
gain = 2.5  # brightness gain for the true color composite


def ndsi_snow(data):
    """
    Classify snow in Sentinel-2 L2A imagery with the NDSI.
    Input data array: [B02, B03, B04, B11]

    Snow pixels are shown in blue.
    Other pixels are shown as a brightened true color composite.
    """
    B02, B03, B04, B11 = data[0], data[1], data[2], data[3]

    NDSI = (B03 - B11) / (B03 + B11)

    snow_mask = NDSI > ndsi_threshold

    red = if_(snow_mask, 0.0, B04 * gain)
    green = if_(snow_mask, 0.8, B03 * gain)
    blue = if_(snow_mask, 1.0, B02 * gain)

    return array_create([red, green, blue])


# Apply the NDSI classification on the data cube on the bands dimension
snow_image = s2cube.apply_dimension(
    dimension=s2cube.metadata.band_dimension.name, process=ndsi_snow
)

# Linear scale to 0-255 for RGB visualization
snow_image = snow_image.linear_scale_range(
    input_min=0, input_max=1, output_min=0, output_max=255
)

graph = snow_image

## Show the result in JupyterGIS

The map has an OpenStreetMap basemap. The openEO layer is added on top of it.
`add_openeo_tile_layer` creates an XYZ service on the backend and fits the map to the bounding box.

The backend calculates each tile on request. A tile can take 10 to 25 seconds, so the layer fills in slowly.

Blue is snow. The other pixels are true color.

In [ ]:
from jupytergis import GISDocument

doc = GISDocument(latitude=-45.355, longitude=168.685, zoom=10)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="© OpenStreetMap contributors",
)

In [ ]:
doc.add_openeo_tile_layer(graph, name="NDSI Snow Classification")
doc

## Technical notes

**Applications:**
- Monitoring seasonal snowpack extent and melt for water resource management.
- Supporting avalanche and flood risk assessment.
- Tracking glacier and mountain snow cover change over time.
- Validating hydrological and climate models.

**Limitations:**
- The mosaic uses the first valid pixel of a short time range. Bright clouds, ice on lakes and some bright artificial surfaces can look like snow and be misclassified.
- Water and shadows can also have a high NDSI. The [snow classifier notebook](snow_classifier_sentinel2.ipynb) adds a brightness check (`B03 > 0.3`) and an NDVI constraint to reduce these errors.
- The fixed NDSI threshold (0.42) may need adjusting for different seasons or snow conditions.

## Attribution

**Original script:** [Normalised Difference Snow Index, NDSI](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/ndsi/) (authors not listed), [Sentinel Hub Custom Scripts](https://github.com/sentinel-hub/custom-scripts)